# CMS Data Access on NRP

**[website version](https://training.nrp-nautilus.io/cms-hats/5_cms_data.html)** — run cells with **Shift+Enter**.

This lesson uses the **NRP USCMS Analysis Hub** ([uscms-af.nrp-nautilus.io](https://uscms-af.nrp-nautilus.io)) — the JupyterHub-based environment introduced as "Method 1" on the [setup page](../../lessons/0_setup.md). If you've been using your own machine for the rest of this training, this is the lesson where you switch over — there's no local install for the tools below. CMS data lives on grid storage (EOS, dCache, and friends) protected by the same CERN grid-certificate infrastructure used across WLCG. `kubectl` (covered on the setup page via `grid-kube-setup`) gets you onto the Nautilus cluster; a **grid certificate** and the short-lived **X.509 proxy** derived from it are what get *you* into CMS data. This lesson sets one up, then uses it to pull a real CMS file and make a plot.

## Setting up your grid certificate

You'll need your CERN grid certificate exported as a `.p12` file. Drag it into the JupyterLab file browser on the left (or use the **Upload** button) before continuing. It only needs to live there for the import step below — you can delete it from the file browser afterward.

### 1. Import the certificate

**🖥️ Terminal step** — prompts for your import password and a PEM pass phrase, so it can't run as a notebook cell. Open a terminal (**File → New → Terminal**):

```bash
grid-cert-import
```

1. **The import password** — the one you chose when exporting the `.p12` file.
2. **A PEM pass phrase** — protects the key at rest on the hub. You'll type it every time you create a proxy, so keeping it the same as the import password is fine.

<details>
<summary>Expected output</summary>

```text
jovyan@jupyter-...:~$ grid-cert-import
Importing: /home/jovyan/myCertificate.p12
  existing usercert.pem -> usercert.pem.20260804150152.bak
  existing userkey.pem -> userkey.pem.20260804150152.bak

Enter Import Password:
Enter Import Password:
Enter PEM pass phrase:
Verifying - Enter PEM pass phrase:

Installed:
  subject=DC=ch, DC=cern, OU=Organic Units, OU=Users, CN=ddiaz, CN=821822, CN=Daniel Diaz
  notBefore=Aug  4 05:43:37 2026 GMT
  notAfter=Sep  8 05:43:37 2027 GMT

Next:   grid-proxy-init
```
</details>

### 2. Create your proxy

**🖥️ Terminal step** — prompts for the PEM pass phrase:

```bash
grid-proxy-init
```

Contacts the CMS VOMS server and writes a short-lived proxy to `~/.globus/x509up`. A `voms2-...` server occasionally times out on the first attempt — `grid-proxy-init` retries automatically, so a retry line in the output is expected, not a failure.

<details>
<summary>Expected output</summary>

```text
jovyan@jupyter-...:~$ grid-proxy-init
Enter GRID pass phrase for this identity:
Contacting  voms2-cms-auth.cern.ch:443 [/DC=ch/DC=cern/OU=computers/CN=voms2-cms-auth.cern.ch] "cms"...
Error contacting  voms-cms-auth.cern.ch:443 for VO cms: voms-cms-auth.cern.ch
Contacting  voms2-cms-auth.cern.ch:443 [/DC=ch/DC=cern/OU=computers/CN=cms-auth.cern.ch] "cms"...
Remote VOMS server contacted succesfully.

Created proxy in /home/jovyan/.globus/x509up.

Your proxy is valid until Wed Aug 12 15:02:24 UTC 2026

  /DC=ch/DC=cern/OU=Organic Units/OU=Users/CN=ddiaz/CN=821822/CN=Daniel Diaz/CN=1734779629
  691198
  cms

Proxy written to /home/jovyan/.globus/x509up
To use it from pods in other namespaces:  grid-proxy-publish <namespace> [...]
```
</details>

![Grid certificate import and proxy creation in a hub terminal](../../images/grid-cert.png)

### 3. Verify it

In [ ]:
xrdcp root://cmsxrootd.fnal.gov//store/group/lpclonglived/B-ParkingLLPs/keep.txt .


If the copy succeeds, your proxy is good — `~/.globus/x509up` is the default location `xrdcp` (and everything below) looks for a proxy, so you don't need to set anything explicitly.

## Access CMS data with uproot

The `xrdcp` step above proved your proxy works. Now use it from Python: pull down one real CMS NanoAOD-style file and plot something from it with [uproot](https://uproot.readthedocs.io/).

Copy the file down with `xrdcp`, same as the verification step above:

In [ ]:
xrdcp -f root://eoscms.cern.ch//eos/cms/store/group/cmst3/group/l1tr/maglowac/AD_HLT_PF/QCD_Bin-Pt-15to7000_TuneCP5_13p6TeV_pythia8/re-emul_Run3Winter25MiniAOD-FEVTOUTPUT_142X_v7-v1/251124_134438/0000/nanoout_1.root nanoout_1.root


Open it with uproot and histogram the jet transverse momenta. The hub image isn't guaranteed to have the HEP Python stack installed, so this installs anything missing into your user site-packages before importing it:

In [ ]:
python3 <<'PY'
import importlib
import subprocess
import sys

for pkg in ("uproot", "awkward", "matplotlib"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "--user", "--quiet", pkg], check=True)

import awkward as ak
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import uproot

events = uproot.open("nanoout_1.root")["Events"]
jet_pt = ak.flatten(events["Jet_pt"].array(entry_stop=5000))

plt.hist(jet_pt, bins=50, range=(0, 200))
plt.xlabel("Jet $p_T$ [GeV]")
plt.ylabel("Jets / bin")
plt.title(f"Jet $p_T$ spectrum ({len(jet_pt)} jets, first 5000 events)")
plt.savefig("jet_pt.png", dpi=150)
print(f"Wrote jet_pt.png from {len(jet_pt)} jets")
PY


Open `jet_pt.png` from the JupyterLab file browser on the left to see the plot. `Events` is the standard NanoAOD tree name; `uproot.open(...).keys()` lists every tree/branch in the file if you want to plot something else (`Muon_pt`, `MET_pt`, and similar branches are also in this file).

## Sharing your proxy with another namespace

Your proxy lives at `~/.globus/x509up` on the hub, but Kubernetes Jobs run in a namespace and can't reach your home directory directly. `grid-proxy-publish` copies your proxy into a namespace as a Secret, so Jobs there can mount it and use it the same way you just used it from the terminal:

In [ ]:
grid-proxy-publish <namespace>   # ✏️ EDIT to your own personal namespace


**⚠️ Only publish to your own personal namespace**

**Never** run `grid-proxy-publish` against a shared or team namespace — only your own personal one.

Any member of a namespace can read that namespace's Secrets. If you publish your proxy to a shared namespace, every other member can use *your* proxy — acting as you against CMS grid storage — without your knowledge. Since a grid proxy is tied to your personal identity and the CERN Certificate Authority's usage policy holds *you* responsible for whatever it's used for, sharing access this way is a policy violation even if nothing goes wrong technically.

If a Job in a shared namespace needs grid data access, have the person who runs that Job publish their **own** proxy there — don't publish yours on their behalf.

## Clean up

Proxies are short-lived by design, so there's nothing to revoke. Remove the file you copied down:

In [ ]:
rm -f nanoout_1.root


If you published a proxy to a namespace you don't want it in anymore, find the Secret it created and delete it:

In [ ]:
kubectl get secrets -n <namespace>   # ✏️ EDIT to the namespace you published to


In [ ]:
kubectl delete secret -n <namespace> <secret-name>   # ✏️ EDIT both placeholders


---

## ✅ Check your work

Verifies the state of your resources on the cluster — rerun any time.

In [ ]:
bash check.sh 5
